# 03 - Forecasting
## Train sales forecasting models, compare models, evaluate metrics, register best model with MLflow

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.lightgbm
import joblib
from pathlib import Path
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
%matplotlib inline

In [ ]:
DATA_DIR = '../data/raw/'
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

transactions = pd.read_csv(f'{DATA_DIR}transaction_data.csv')
products = pd.read_csv(f'{DATA_DIR}product.csv')

df = transactions.merge(products[['PRODUCT_ID', 'DEPARTMENT']], on='PRODUCT_ID', how='left')
df = df.sort_values(['PRODUCT_ID', 'DAY']).reset_index(drop=True)

In [ ]:
# Build features
def build_features(df):
    result = pd.DataFrame(index=df.index)
    
    result['day_of_week'] = df['DAY'] % 7
    result['month'] = ((df['DAY'] / 30).astype(int) % 12) + 1
    result['quarter'] = ((df['DAY'] / 30).astype(int) % 12 // 3) + 1
    result['week_of_year'] = (df['DAY'] / 7).astype(int) % 52
    result['is_weekend'] = (result['day_of_week'] >= 5).astype(int)
    
    result['retail_disc'] = df['RETAIL_DISC'].abs()
    result['coupon_disc'] = df['COUPON_DISC'].abs()
    result['total_discount'] = result['retail_disc'] + result['coupon_disc']
    result['has_discount'] = (result['total_discount'] > 0).astype(int)
    
    for lag in [1, 7, 14]:
        result[f'lag_{lag}'] = df.groupby('PRODUCT_ID')['QUANTITY'].shift(lag).fillna(0)
    
    for window in [7, 14]:
        result[f'rolling_mean_{window}'] = (
            df.groupby('PRODUCT_ID')['QUANTITY']
            .transform(lambda x: x.rolling(window, min_periods=1).mean().shift(1))
            .fillna(0)
        )
    
    result.fillna(0, inplace=True)
    return result

In [ ]:
# Use top 50 products by volume
top_products = df.groupby('PRODUCT_ID')['QUANTITY'].sum().nlargest(50).index
sample_df = df[df['PRODUCT_ID'].isin(top_products)].copy()
sample_df = sample_df.sort_values(['PRODUCT_ID', 'DAY']).reset_index(drop=True)

features = build_features(sample_df)
target = sample_df['QUANTITY'].values

feature_cols = [c for c in features.columns]
X = features[feature_cols].values
y = target

print(f'Training samples: {len(X):,}')
print(f'Feature count: {len(feature_cols)}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')

In [ ]:
# Compare models
models = {
    'lightgbm': LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42, verbose=-1),
    'lightgbm_deep': LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=10, num_leaves=64, random_state=42, verbose=-1),
    'lightgbm_shallow': LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42, verbose=-1),
    'random_forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    print(f'{name:25s}  MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.4f}')

In [ ]:
# Select best model
best_model_name = min(results, key=lambda k: results[k]['MAE'])
best_model = models[best_model_name]
print(f'Best model: {best_model_name}')
print(f'Results: {results[best_model_name]}')

In [ ]:
# Plot predictions vs actual
y_pred = best_model.predict(X_test)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.scatter(y_test[:500], y_pred[:500], alpha=0.5)
plt.plot([0, y_test[:500].max()], [0, y_test[:500].max()], 'r--')
plt.xlabel('Actual Quantity')
plt.ylabel('Predicted Quantity')
plt.title(f'{best_model_name}: Predictions vs Actual')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
errors = y_test - y_pred
plt.hist(errors, bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Prediction Error')
plt.ylabel('Frequency')
plt.title(f'Error Distribution (MAE={results[best_model_name]["MAE"]:.3f})')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Log to MLflow and save model
mlflow.set_tracking_uri(f'file://{MODELS_DIR / "mlruns"}')
mlflow.set_experiment('promotion_intelligence')

with mlflow.start_run(run_name=f'{best_model_name}_forecast'):
    mlflow.log_params({
        'model_type': best_model_name,
        'n_samples': len(X_train),
        'feature_count': len(feature_cols),
        'feature_cols': str(feature_cols),
    })
    
    mlflow.log_metrics({
        'mae': results[best_model_name]['MAE'],
        'rmse': results[best_model_name]['RMSE'],
        'r2': results[best_model_name]['R2'],
    })
    
    mlflow.lightgbm.log_model(best_model, artifact_path='forecast_model')
    run_id = mlflow.active_run().info.run_id
    
    local_path = MODELS_DIR / 'forecast_model.pkl'
    joblib.dump(best_model, local_path)
    
    import json
    with open(MODELS_DIR / 'forecast_model_meta.json', 'w') as f:
        json.dump({
            'run_id': run_id,
            'model_type': best_model_name,
            'feature_cols': feature_cols,
            'mae': results[best_model_name]['MAE'],
            'rmse': results[best_model_name]['RMSE'],
        }, f)

print(f'Model saved to {MODELS_DIR / "forecast_model.pkl"}')
print(f'MLflow run_id: {run_id}')

In [ ]:
# Verify model can be loaded
loaded = joblib.load(MODELS_DIR / 'forecast_model.pkl')
test_pred = loaded.predict(X_test[:5])
print('Model verification - predictions:', test_pred)